<a href="https://colab.research.google.com/github/jms011208/zero_DCE/blob/improve%2Fdenoise-net/zero_DCE_clean.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Zero-DCE 저조도 밝기 개선 + 노이즈 제거 비교 (실행용)

위에서 아래로 순서대로 실행하면 되는 정리본입니다. `런타임 → 모두 실행`도 가능합니다.
시행착오까지 담긴 실행 기록은 `zero_DCE.ipynb`(기록용)에 있습니다.

**시작 전**: `런타임 → 런타임 유형 변경 → T4 GPU`

| 단계 | 내용 | 예상 시간 (T4) |
|---|---|---|
| 0 | 설정 | - |
| 1 | 저장소와 데이터 준비 (다시 실행해도 안전) | 약 2분 |
| 2 | 원본 Zero-DCE 결과 (기준) | 1분 미만 |
| 3 | 노이즈 억제 CNN 학습 2종 (`denoise_v1`, `denoise_edge4`) | 학습 시간 × 2 |
| 4 | LOL eval15 PSNR/SSIM 평가 | 1분 미만 |
| 5 | 결과 이미지 비교 (기존 vs CNN) | 1분 미만 |
| 6 | 노이즈 제거 방법 비교 (CNN vs 고정 수식 필터) | 약 2분 (BM3D 포함 시 +10분) |
| 7 | 결과 내려받기 (선택) | - |

지표와 방법 설명은 `zero_DCEob/05 지표 평가 정리.md` 참고.

## 0. 설정

In [ ]:
BRANCH = "improve/denoise-net"
EPOCHS = 30                 # 노이즈 억제 CNN 학습 epoch
USE_DRIVE = False           # True: 학습 가중치를 구글 드라이브에 저장/복원 (런타임이 끊겨도 재학습 불필요)
DRIVE_DIR = "/content/drive/MyDrive/zero_DCE_weights"
RUN_BM3D = False            # True: 6단계 비교에 BM3D 포함 (15장 약 10분 추가)
SHOW_IMAGE = "179"          # 5, 6단계에서 확대해 볼 LOL 이미지 번호

## 1. 저장소와 데이터 준비
이미 받은 저장소와 데이터는 건너뛰므로 여러 번 실행해도 폴더가 중첩되지 않습니다.

In [ ]:
import os
os.chdir("/content")
if not os.path.isdir("/content/zero_DCE/.git"):
    !git clone -q -b {BRANCH} https://github.com/jms011208/zero_DCE.git /content/zero_DCE
os.chdir("/content/zero_DCE")
!git checkout -q {BRANCH} && git pull -q
!git log --oneline -1
print("작업 폴더:", os.getcwd())

In [ ]:
os.chdir("/content/zero_DCE")
os.makedirs("data", exist_ok=True)

# 학습용 데이터 (73MB, 2,002장)
if not os.path.isdir("data/train_data"):
    !pip install -q gdown
    !gdown 1GAB3uGsmAyLgtDBDONbil08vVu5wJcG3 -O /content/train_data.zip
    !unzip -q /content/train_data.zip -d data/

# 평가용 LOL (347MB, eval15 15쌍 / our485 485쌍)
if not os.path.isdir("data/LOL"):
    !wget -q -O /content/lol.zip https://huggingface.co/datasets/geekyrakshit/LoL-Dataset/resolve/main/lol_dataset.zip
    !unzip -q /content/lol.zip -d data/ && mv data/lol_dataset data/LOL

# 정성 평가용 테스트 이미지 (DICM, LIME)
if not os.path.isdir("data/test_data"):
    !git clone -q --depth 1 https://github.com/Li-Chongyi/Zero-DCE.git /tmp/zdce
    !cp -r /tmp/zdce/Zero-DCE_code/data/test_data data/

if RUN_BM3D:
    !pip install -q bm3d

In [ ]:
n_train = len([f for f in os.listdir("data/train_data") if f.lower().endswith(".jpg")])
print("train_data :", n_train, "장 (2002이어야 함)")
print("LOL eval15 :", [d for d in sorted(os.listdir("data/LOL/eval15")) if not d.startswith(".")])
print("test_data  :", sorted(os.listdir("data/test_data")))
assert n_train == 2002, "학습 데이터가 없습니다. 위 셀의 gdown 출력(다운로드 한도 등)을 확인하세요."

## 2. 원본 Zero-DCE로 밝기 개선 (기준)
`data/test_data`의 이미지를 원본 가중치 `snapshots/Epoch99.pth`로 처리해 `data/result/`에 저장합니다.

In [ ]:
!python lowlight_test.py > /dev/null && echo "완료: data/result/"

In [ ]:
import matplotlib.pyplot as plt, matplotlib.image as mpimg
name = "DICM/04.jpg"
plt.figure(figsize=(12, 5))
for i, (t, p) in enumerate([("input", f"data/test_data/{name}"), ("Zero-DCE", f"data/result/{name}")]):
    plt.subplot(1, 2, i + 1); plt.imshow(mpimg.imread(p)); plt.title(t); plt.axis("off")
plt.show()

## 3. 노이즈 억제 CNN 학습
Zero-DCE(`Epoch99.pth`)는 고정하고, 뒤에 붙인 작은 CNN만 합성 노이즈로 학습합니다.
- `denoise_v1`: 기본 손실
- `denoise_edge4`: 경계 보존 손실 추가 (`--lam_edge 4`)

가중치가 이미 있으면(드라이브에서 복원했거나 앞서 학습한 경우) 학습을 건너뜁니다.

In [ ]:
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    for name in ["denoise_v1", "denoise_edge4"]:
        if os.path.isdir(f"{DRIVE_DIR}/{name}") and not os.path.exists(f"snapshots/{name}/denoiser_latest.pth"):
            !mkdir -p snapshots && cp -r "{DRIVE_DIR}/{name}" snapshots/
            print("드라이브에서 복원:", name)

In [ ]:
for name, extra in [("denoise_v1", ""), ("denoise_edge4", "--lam_edge 4")]:
    if os.path.exists(f"snapshots/{name}/denoiser_latest.pth"):
        print(f"[건너뜀] {name}: 가중치가 이미 있습니다")
        continue
    print(f"===== {name} 학습 시작 =====")
    !python lowlight_train_v2.py --num_epochs {EPOCHS} {extra} --snapshots_folder snapshots/{name}/
    if USE_DRIVE:
        !mkdir -p "{DRIVE_DIR}" && cp -r snapshots/{name} "{DRIVE_DIR}/"
        print("드라이브에 저장:", name)

## 4. LOL eval15 평가 (PSNR/SSIM)
기존 Zero-DCE와 노이즈 억제 CNN을 정답 이미지와 비교합니다. PSNR은 밝기 차이의 영향이 커서 SSIM을 같이 보세요.

In [ ]:
for name in ["denoise_v1", "denoise_edge4"]:
    print(f"===== {name} =====")
    !python eval_lol.py --lol_dir data/LOL/eval15 --denoise_weights snapshots/{name}/denoiser_latest.pth

## 5. 결과 이미지 비교 (기존 vs CNN)
LOL eval15 저조도 이미지에 두 가중치를 적용합니다. `*_base.png`는 기존 Zero-DCE 결과, `*_mask.png`는 노이즈 마스크입니다.

In [ ]:
for name in ["denoise_v1", "denoise_edge4"]:
    !python lowlight_test_v2.py --input data/LOL/eval15/low --output data/result_{name}/ --denoise_weights snapshots/{name}/denoiser_latest.pth --save_extra > /dev/null
print("완료: data/result_denoise_v1/, data/result_denoise_edge4/")

In [ ]:
n = SHOW_IMAGE
items = [("input", f"data/LOL/eval15/low/{n}.png"),
         ("Zero-DCE (base)", f"data/result_denoise_v1/{n}_base.png"),
         ("+ CNN (v1)", f"data/result_denoise_v1/{n}.png"),
         ("+ CNN (edge4)", f"data/result_denoise_edge4/{n}.png"),
         ("ground truth", f"data/LOL/eval15/high/{n}.png"),
         ("noise mask (edge4)", f"data/result_denoise_edge4/{n}_mask.png")]
fig, ax = plt.subplots(1, len(items), figsize=(5 * len(items), 4))
for a, (t, p) in zip(ax, items):
    im = mpimg.imread(p); a.imshow(im, cmap="gray" if im.ndim == 2 else None); a.set_title(t); a.axis("off")
plt.show()

## 6. 노이즈 제거 방법 비교 (CNN vs 고정 수식 필터)
같은 Zero-DCE 결과에 각 방법을 적용해 같은 지표로 비교합니다.
- `edge_ratio`: 경계 유지율 (1=유지, 작을수록 흐려짐)
- `flat_luma`, `flat_chroma`: 평평한 영역의 밝기/색 노이즈 (낮을수록 좋음)
- `ycc_*`: 밝기는 Zero-DCE 결과를 유지하고 색 노이즈만 필터로 제거 (`--ycc_luma`로 밝기에 필터를 섞는 비율 조절)

**6-1. 필터 기본값**

In [ ]:
CNN = "snapshots/denoise_edge4/denoiser_latest.pth"
methods = "base,cnn,nlm,bilateral,guided" + (",bm3d" if RUN_BM3D else "") + ",ycc_nlm,ycc_bilateral,ycc_guided"
!python compare_denoise.py --methods {methods} --cnn_weights {CNN} --csv data/cmp_default.csv | grep -v "^done:"

**6-2. 필터 약하게** (경계 유지율을 CNN과 비슷하게 맞춘 공정 비교)

In [ ]:
!python compare_denoise.py --methods base,cnn,nlm,bilateral,guided --cnn_weights {CNN} --nlm_h 3 --nlm_h_color 8 --bil_sigma_color 0.05 --gf_eps 0.001 --csv data/cmp_weak.csv | grep -v "^done:"

**6-3. 밝기/색 분리, 밝기 30% 섞기** (결과 이미지도 저장)

In [ ]:
!python compare_denoise.py --methods base,cnn,nlm,bilateral,ycc_nlm,ycc_bilateral --ycc_luma 0.3 --cnn_weights {CNN} --save_dir data/result_ycc03 --csv data/cmp_ycc03.csv | grep -v "^done:"

**6-4. 확대 비교** (경계와 질감이 보이는 위치로 `y0, x0`를 조정하세요. 이미지는 400x600)

In [ ]:
n, y0, x0, s = SHOW_IMAGE, 150, 250, 120
methods = ["base", "cnn", "nlm", "bilateral", "ycc_nlm", "ycc_bilateral"]
fig, ax = plt.subplots(1, len(methods) + 1, figsize=(4 * (len(methods) + 1), 4.5))
for a, m in zip(ax, methods):
    a.imshow(mpimg.imread(f"data/result_ycc03/{m}/{n}.png")[y0:y0 + s, x0:x0 + s], interpolation="nearest")
    a.set_title(m); a.axis("off")
ax[-1].imshow(mpimg.imread(f"data/LOL/eval15/high/{n}.png")[y0:y0 + s, x0:x0 + s], interpolation="nearest")
ax[-1].set_title("ground truth"); ax[-1].axis("off")
plt.show()

## 7. 결과 내려받기 (선택)
런타임이 끊기면 결과가 사라지므로 필요한 결과는 내려받거나 드라이브에 복사하세요.

In [ ]:
!zip -qr /content/zero_dce_results.zip data/result data/result_denoise_v1 data/result_denoise_edge4 data/result_ycc03 data/cmp_*.csv
from google.colab import files
files.download("/content/zero_dce_results.zip")